# Headline

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Download

In [10]:
!pip install vnstock

!pip install ta
!pip install arch

## Import

In [11]:
from vnstock import *
import numpy as np
import pandas as pd

import ta  # Technical Analysis library
from arch import arch_model # For GARCH
from scipy.stats import entropy
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA # Added for ARIMA model

# Data Input

In [12]:
quote = Quote(symbol='DIG', source='KBS')
raw_historical_data = quote.history(start='2000-01-01', end='2025-05-25')

'''
data = raw_historical_data.drop(['time'], axis=1)
'''
data = raw_historical_data.set_index('time')
data

,open,high,low,close,volume
time,,,,,
2015-11-26 07:00:00,4.34,4.47,4.30,4.30,469900
2015-11-27 07:00:00,4.30,4.34,4.26,4.30,724040
2015-11-30 07:00:00,4.30,4.30,4.18,4.22,418830
2015-12-01 07:00:00,4.26,4.26,4.13,4.13,464040
2015-12-02 07:00:00,4.18,4.26,4.18,4.18,171580
...,...,...,...,...,...
2025-05-19 07:00:00,13.54,14.10,13.45,13.84,15060300
2025-05-20 07:00:00,13.93,13.97,13.67,13.71,6530700
2025-05-21 07:00:00,13.80,13.80,13.50,13.67,8952900


In [13]:
from statsmodels.tsa.stattools import adfuller

def print_adf_results(data):
    result = adfuller(data)
    print('ADF Statistic: %f' % result[0])
    print('p-value: %f' % result[1])
    print('Critical Values:')
    for key, value in result[4].items():
        print('\t%s: %.3f' % (key, value))

#print_adf_results(np.log(data.loc[:, ['Close']]).diff().dropna())
print_adf_results(data.loc[:, ['close']].diff().dropna())

ADF Statistic: -8.974806
p-value: 0.000000
Critical Values:
	1%: -3.433
	5%: -2.863
	10%: -2.567


# Data Preparation:

## 1. HELPER FUNCTIONS (The "Math" Engine)

In [14]:
def get_hurst_exponent(time_series, max_lag=20):
    """Returns the Hurst Exponent of the time series vector ts"""
    lags = range(2, max_lag)
    tau = [np.sqrt(np.std(np.subtract(time_series[lag:], time_series[:-lag]))) for lag in lags]
    poly = np.polyfit(np.log(lags), np.log(tau), 1)
    return poly[0] * 2.0

def calculate_approx_entropy(U, m=2, r=0.2):
    """Calculates Approximate Entropy (complexity of the series)"""
    def _maxdist(x_i, x_j):
        return max([abs(ua - ub) for ua, ub in zip(x_i, x_j)])

    def _phi(m):
        x = [[U[j] for j in range(i, i + m - 1 + 1)] for i in range(N - m + 1)]
        C = [len([1 for x_j in x if _maxdist(x_i, x_j) <= r]) / (N - m + 1.0) for x_i in x]
        return (N - m + 1.0)**(-1) * sum(np.log(C))

    N = len(U)
    return abs(_phi(m+1) - _phi(m))

## 2. FEATURE GENERATION CLASSES

In [15]:
class FeatureEngineer:
    def __init__(self, df):
        # Expects df with: ['Open', 'High', 'Low', 'Close', 'Volume']
        self.df = df.copy()

    def add_technical_indicators(self):
        """Adds ~80+ standard technical indicators using the 'ta' library."""
        print("Generating Technical Indicators...")

        # 1. Store the original price/target columns separately
        original_cols = ['Open', 'High', 'Low', 'Close', 'Volume']


        # 2. This single line adds RSI, MACD, Bollinger, Stochastic, Ichimoku, etc.
        self.df = ta.add_all_ta_features(
            self.df, open="Open", high="High", low="Low", close="Close", volume="Volume", fillna=True
        )

        # 3. IDENTIFY THE NEW FEATURES (exclude original prices)
        new_features = [col for col in self.df.columns if col not in original_cols]

        # This ensures that for the row representing 'Today',
        # the indicators are actually based on data from 'Yesterday'.
        self.df[new_features] = self.df[new_features].shift(1)

        return self

    def add_statistical_features(self, window=20):
        """Adds statistical properties: Rolling Z-Score, Skew, Kurtosis, Hurst."""
        print("Generating Statistical Features...")

        # Log Returns (The base for stationarity)
        self.df['log_ret'] = np.log(self.df['Close'] / self.df['Close'].shift(1))
        self.df['ret'] = self.df['Close'] - self.df['Close'].shift(1)

        # Rolling Statistical Moments
        self.df[f'roll_std_{window}'] = self.df['log_ret'].rolling(window=window).std()
        self.df[f'roll_skew_{window}'] = self.df['log_ret'].rolling(window=window).skew()
        self.df[f'roll_kurt_{window}'] = self.df['log_ret'].rolling(window=window).kurt()

        # Z-Score (Distance from Mean in std devs)
        rolling_mean = self.df['Close'].rolling(window=window).mean()
        rolling_std = self.df['Close'].rolling(window=window).std()
        self.df['z_score'] = (self.df['Close'] - rolling_mean) / rolling_std

        # Rolling Hurst Exponent (Vectorized approach is hard, using apply for demonstration)
        # Note: In production, optimize this as it is slow on large dataframes
        self.df['hurst'] = self.df['Close'].rolling(window=100).apply(lambda x: get_hurst_exponent(x.values))

        return self

    def add_garch_volatility(self):
        """Adds GARCH(1,1) conditional volatility forecast."""
        print("Fitting GARCH Model...")
        # We need a clean series without NaNs for GARCH
        returns = self.df['log_ret'].dropna() * 100 # Rescale for better convergence

        # Fit the ARIMA model to get residuals for GARCH
        # ARIMA(p,d,q) where d=0 as we are providing log returns directly
        order = (1, 0, 2)
        model_arima = ARIMA(returns, order=order)
        model_arima_fit = model_arima.fit()

        # Fit GARCH(1,1) using the residuals from the ARIMA model
        model_arch = arch_model(
            model_arima_fit.resid,
            mean='Zero',
            vol='Garch',
            p=1, # ARCH order for variance
            q=1, # GARCH order for variance
            dist='t'
        )
        res = model_arch.fit(disp='off')

        # Get the conditional volatility (sigma)
        self.df.loc[returns.index, 'garch_volatility'] = res.conditional_volatility
        return self

    def add_microstructure_proxies(self):
        """Adds Volume/Liquidity proxies since we lack L2 data."""
        print("Generating Microstructure Proxies...")

        # 1. VWAP (Volume Weighted Average Price)
        self.df['vwap'] = (self.df['Volume'] * (self.df['High'] + self.df['Low'] + self.df['Close']) / 3).cumsum() / self.df['Volume'].cumsum()

        # 2. VWAP Distance (Trend Signal)
        self.df['dist_vwap'] = (self.df['Close'] - self.df['vwap']) / self.df['vwap']

        # 3. Force Index (Volume * Price Change)
        self.df['force_index'] = self.df['Close'].diff(1) * self.df['Volume']

        # 4. Amihud Illiquidity Ratio Proxy (Absolute Return / Volume)
        # High value = low liquidity (price moves a lot on little volume)
        self.df['amihud_illiq'] = self.df['log_ret'].abs() / self.df['Volume']

        return self

    def add_target_and_clean(self):
        """Creates target variable and handles NaNs."""
        # Target: 1 if price goes UP in next period, 0 otherwise
        #self.df['target'] = np.where(self.df['Close'].shift(-1) > self.df['Close'], 1, 0)

        # Drop initial NaNs generated by rolling windows
        self.df.dropna(inplace=True)
        return self.df

## 3. EXECUTION

In [16]:
# Mock Data Generation (Replace this with your VNM.csv load)
data.columns = ['Open', 'High', 'Low', 'Close', 'Volume']

# Run the Pipeline
engineer = FeatureEngineer(data.copy())
engineer.add_technical_indicators()
engineer.add_statistical_features(window=20)
engineer.add_garch_volatility()
engineer.add_microstructure_proxies()

df_final = engineer.add_target_and_clean()

print(f"\nFinal Feature Set Shape: {df_final.shape}")
print("Sample Features:", df_final.columns[:10].tolist() + ["..."])
print(f"GARCH Volatility Head:\n{df_final[['log_ret', 'garch_volatility']].head()}")

Generating Technical Indicators...


Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`


Generating Statistical Features...
Fitting GARCH Model...


A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.


Generating Microstructure Proxies...

Final Feature Set Shape: (2269, 103)
Sample Features: ['Open', 'High', 'Low', 'Close', 'Volume', 'volume_adi', 'volume_obv', 'volume_cmf', 'volume_fi', 'volume_em', '...']
GARCH Volatility Head:
                      log_ret  garch_volatility
time                                           
2016-04-22 07:00:00  0.060977          2.173286
2016-04-25 07:00:00  0.033244          2.807469
2016-04-26 07:00:00  0.013532          2.801939
2016-04-27 07:00:00 -0.035571          2.674291
2016-04-28 07:00:00  0.000000          2.858598


# File Saving:

In [20]:
df_final.to_csv('/content/drive/MyDrive/COURSES/Extension Projects/Code/Data Sources/DIG_data_set_for_regimes.csv', index=True)

In [22]:
df = pd.read_csv('/content/drive/MyDrive/COURSES/Extension Projects/Code/Data Sources/DIG_data_set_for_regimes.csv')

df

,time,Open,High,Low,Close,Volume,volume_adi,volume_obv,volume_cmf,volume_fi,...,roll_std_20,roll_skew_20,roll_kurt_20,z_score,hurst,garch_volatility,vwap,dist_vwap,force_index,amihud_illiq
0,2016-04-22 07:00:00,3.38,3.55,3.34,3.55,2761460,-9.616120e+06,8.714330e+06,-0.165504,16968.270979,...,0.025241,0.864960,0.310577,2.673555,0.334259,2.173286,3.458107,0.026573,579906.6,2.208136e-08
1,2016-04-25 07:00:00,3.67,3.72,3.59,3.67,2336410,-6.854660e+06,1.147579e+07,0.081603,97388.032267,...,0.025637,0.623834,-0.172504,2.649854,0.341582,2.807469,3.468953,0.057956,280369.2,1.422869e-08
2,2016-04-26 07:00:00,3.67,3.76,3.63,3.72,1204680,-6.315488e+06,1.381220e+07,0.105778,123528.199086,...,0.025612,0.539774,-0.235840,2.331197,0.357664,2.801939,3.475270,0.070420,60234.0,1.123286e-08
3,2016-04-27 07:00:00,3.72,3.76,3.59,3.59,1570480,-5.852150e+06,1.501688e+07,0.128934,114486.170645,...,0.026388,0.413794,-0.224038,1.492845,0.373076,2.674291,3.481088,0.031287,-204162.4,2.265006e-08
4,2016-04-28 07:00:00,3.59,3.67,3.55,3.59,746480,-7.422630e+06,1.344640e+07,0.038115,68964.946267,...,0.026388,0.413794,-0.224038,1.333182,0.389223,2.858598,3.483029,0.030712,0.0,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2264,2025-05-19 07:00:00,13.54,14.10,13.45,13.84,15060300,-6.929636e+08,1.772220e+09,-0.260383,131040.816306,...,0.023348,-1.697352,4.326719,1.282793,0.583102,2.353832,21.121610,-0.344747,3162663.0,1.015232e-09
2265,2025-05-20 07:00:00,13.93,13.97,13.67,13.71,6530700,-6.899516e+08,1.787280e+09,-0.191036,564129.699691,...,0.022613,-1.687347,4.493780,0.968153,0.586032,2.299141,21.118543,-0.350808,-848991.0,1.445091e-09
2266,2025-05-21 07:00:00,13.80,13.80,13.50,13.67,8952900,-6.947408e+08,1.780750e+09,-0.295668,362255.314020,...,0.022626,-1.662990,4.421019,0.859906,0.589697,2.224058,21.114271,-0.352571,-358116.0,3.263571e-10
2267,2025-05-22 07:00:00,13.67,13.93,13.50,13.76,12181700,-6.935470e+08,1.771797e+09,-0.295142,259345.126303,...,0.021814,-2.001291,6.211229,0.965136,0.591952,2.123972,21.108523,-0.348131,1096353.0,5.386918e-10
